In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/roman-urdu/Roman Urdu reviews Dataset with English translation.csv


In [8]:
!pip install -q transformers datasets imbalanced-learn scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
import pandas as pd
import numpy as np
import re
import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from datasets import Dataset as HFDataset

In [10]:
df = pd.read_csv(
    "/kaggle/input/roman-urdu/Roman Urdu reviews Dataset with English translation.csv",
    encoding="latin-1"
)

df = df.rename(columns={
    "ROMAN URDU REVIEWS": "text",
    "SENTIMENT": "label"
})

In [11]:
def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\S+", "", t)
    t = re.sub(r"[^a-z0-9\s]", "", t)
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t.strip()

df["text"] = df["text"].apply(clean_text)

In [12]:
df["label"] = df["label"].astype(str).str.lower().str.strip()

def map_sentiment(x):
    if x in ["positive", "very positive", "verypositive"]:
        return "positive"
    if x in ["negative", "very negative", "verynegative", "neative"]:
        return "negative"
    return "neutral"

df["label"] = df["label"].apply(map_sentiment)

In [13]:
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["labels"] = df["label"].map(label_map)

In [14]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["labels"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["labels"], random_state=42
)

In [15]:
ros = RandomOverSampler(random_state=42)

X_res, y_res = ros.fit_resample(
    train_df[["text"]],
    train_df["labels"]
)

train_df = pd.DataFrame({
    "text": X_res["text"],
    "labels": y_res
})

In [16]:
tokenizer = AutoTokenizer.from_pretrained(
    "cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [19]:
from datasets import Dataset as HFDataset, Value

# This ensures 'Value' refers to the HuggingFace data type object

In [20]:
# Create HF Datasets from Pandas DataFrames
train_ds = HFDataset.from_pandas(train_df, preserve_index=False)
val_ds   = HFDataset.from_pandas(val_df[["text","labels"]], preserve_index=False)
test_ds  = HFDataset.from_pandas(test_df[["text","labels"]], preserve_index=False)

# Cast 'labels' to int64 for the model's loss function
train_ds = train_ds.cast_column("labels", Value("int64"))
val_ds   = val_ds.cast_column("labels", Value("int64"))
test_ds  = test_ds.cast_column("labels", Value("int64"))

print("Datasets created and columns cast to int64 successfully.")

Casting the dataset:   0%|          | 0/27111 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2809 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2809 [00:00<?, ? examples/s]

Datasets created and columns cast to int64 successfully.


In [21]:
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_ds   = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
test_ds  = test_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

Map:   0%|          | 0/27111 [00:00<?, ? examples/s]

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    num_labels=3,
    ignore_mismatched_sizes=True
).to(device)

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

In [24]:
weights = compute_class_weight(
    "balanced",
    classes=np.unique(train_df["labels"]),
    y=train_df["labels"]
)

weights = torch.tensor(weights, dtype=torch.float).to(device)

In [25]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs["labels"].long()
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [26]:
for param in model.roberta.parameters():
    param.requires_grad = False

print("Stage-1: Encoder frozen")

Stage-1: Encoder frozen


In [28]:
stage1_args = TrainingArguments(
    output_dir="./stage1_model",
    eval_strategy="epoch",       # Fixed: changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none"
)

In [30]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels").long()
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Apply weights to CrossEntropy
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

In [31]:
trainer_stage1 = WeightedTrainer(
    model=model,
    args=stage1_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=lambda p: {
        "f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")
    },
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer_stage1.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,0.984800,0.926760,0.559898
2,0.948100,0.906043,0.562417
3,0.946200,0.906186,0.565127


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=2544, training_loss=0.9649841800425787, metrics={'train_runtime': 886.1561, 'train_samples_per_second': 91.782, 'train_steps_per_second': 2.871, 'total_flos': 5349950901017856.0, 'train_loss': 0.9649841800425787, 'epoch': 3.0})

In [32]:
for param in model.parameters():
    param.requires_grad = True

print("Stage-2: Encoder unfrozen")

Stage-2: Encoder unfrozen


In [33]:
stage2_args = TrainingArguments(
    output_dir="./stage2_model",
    eval_strategy="epoch",       # Fixed: changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none"
)

In [34]:
trainer_stage2 = WeightedTrainer(
    model=model,
    args=stage2_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=lambda p: {
        "f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")
    },
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer_stage2.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,0.767200,0.695309,0.680168
2,0.587600,0.675228,0.686094
3,0.515300,0.708611,0.699082
4,0.473700,0.765416,0.699548
5,0.385300,0.842974,0.715358
6,0.313000,0.964726,0.727423
7,0.245900,0.986933,0.719841
8,0.236300,1.006513,0.720260
9,0.189400,1.195815,0.714723


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=7632, training_loss=0.41833520916522926, metrics={'train_runtime': 4657.1405, 'train_samples_per_second': 116.428, 'train_steps_per_second': 3.642, 'total_flos': 1.6049852703053568e+16, 'train_loss': 0.41833520916522926, 'epoch': 9.0})

In [35]:
out = trainer_stage2.predict(test_ds)
preds = out.predictions.argmax(-1)

print("Final Accuracy:", accuracy_score(out.label_ids, preds))
print("Final Macro-F1:", f1_score(out.label_ids, preds, average="macro"))
print("Confusion Matrix:\n", confusion_matrix(out.label_ids, preds))

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Final Accuracy: 0.740121039515842
Final Macro-F1: 0.7238913376054265
Confusion Matrix:
 [[826 124  85]
 [158 397  90]
 [161 112 856]]


In [36]:
trainer_stage2.save_model("./final_two_stage_model")
tokenizer.save_pretrained("./final_two_stage_model")
print("Two-stage model saved")

Two-stage model saved
